In [2]:
from kafka import KafkaProducer
import json
import time
import random
from math import cos, pi

In [3]:
def generate_sensor_data():
    timestamp = int(time.time())

    # Simulate IoT sensor data for water quality metrics with realistic patterns
    water_temperature = random.uniform(1, 3) + round(20 + 5 * (1 + 0.5 * (1 + cos((timestamp % 86400) / 86400 * 2 * pi))), 2)
    ph_level = random.uniform(0, 1) + round(7.5 + 0.2 * (1 + cos((timestamp % 86400) / 86400 * 2 * pi)), 2)
    turbidity = round(random.uniform(5, 50), 2)  # Turbidity in NTU (Nephelometric Turbidity Units)
    dissolved_oxygen = round(random.uniform(5, 12), 2)  # Dissolved Oxygen in mg/L

    return {
        "timestamp": timestamp,
        "water_temperature": water_temperature,
        "ph_level": ph_level,
        "turbidity": turbidity,
        "dissolved_oxygen": dissolved_oxygen
    }

In [ ]:
####################
# Codigo original:
####################
# Kafka configuration
kafka_topic = "water_quality"
kafka_bootstrap_servers = ["localhost:9092"] 

# Create Kafka producer
# producer = KafkaProducer(
#     bootstrap_servers=kafka_bootstrap_servers,
#     value_serializer=lambda v: json.dumps(v).encode('utf-8')
# )

###########
# Modificacion para varios productores
###########

producer_number = 7 # Elegimos un número de producers (paralelismo + robustez)
producers = [] # Aqui almacenaremos nuestros producers
producer_ids = [] # Le damos a cada uno un ID unico
# Create Kafka producer
for i in range(producer_number):
	producer_ids.append("producerID_"+str(i)) # Generacion del ID
	producers.append(KafkaProducer(
			bootstrap_servers=kafka_bootstrap_servers,
			value_serializer=lambda v: json.dumps(v).encode('utf-8')
			)) # Instanciamos el producers


try:
    while True:
        # Iteramos por cada producer (simulación de múltiples fuentes de datos)
        for i, producer  in enumerate(producers):
            # Generate sensor data
            sensor_data = generate_sensor_data()

            # Publish sensor data to Kafka
            # Añadimos identificador al mensaje
            sensor_data["producer_id"] = producer_ids[i]
            # Añadimos una key única (producer_id) para identificar el origen del mensaje
            producer.send(kafka_topic, key=producer_ids[i].encode('utf-8'), value=sensor_data)

            print(f"Sent: {sensor_data}")

        # Wait for 1 second
        time.sleep(random.uniform(0.5, 1.5))
except KeyboardInterrupt:
    print("Stopped producing messages.")
finally:
    # Cerramos todos los producers
    [producer.close() for producer in producers]

Sent: {'timestamp': 1774026266, 'water_temperature': 28.27502595324867, 'ph_level': 7.799011039068143, 'turbidity': 5.32, 'dissolved_oxygen': 7.56, 'producer_id': 'producerID_0'}
Sent: {'timestamp': 1774026266, 'water_temperature': 29.88117156354325, 'ph_level': 8.468119245488133, 'turbidity': 29.09, 'dissolved_oxygen': 5.6, 'producer_id': 'producerID_1'}
Sent: {'timestamp': 1774026266, 'water_temperature': 28.023847174359947, 'ph_level': 7.890092986396155, 'turbidity': 32.45, 'dissolved_oxygen': 8.68, 'producer_id': 'producerID_2'}
Sent: {'timestamp': 1774026266, 'water_temperature': 29.019831186921373, 'ph_level': 8.283322392431469, 'turbidity': 39.37, 'dissolved_oxygen': 5.16, 'producer_id': 'producerID_3'}
Sent: {'timestamp': 1774026266, 'water_temperature': 29.528108430273363, 'ph_level': 8.558454381121956, 'turbidity': 40.38, 'dissolved_oxygen': 8.74, 'producer_id': 'producerID_4'}
Sent: {'timestamp': 1774026266, 'water_temperature': 28.041432693697075, 'ph_level': 8.595176080765